# Beginner track: potential energy surface of HeH$^+$

**Goal:** compute the total ground-state energy of HeH$^+$ as a function of the He–H distance, plot the curve, find the equilibrium bond length, and describe what happens as the atoms are pulled apart. Also check whether the ansatz stays accurate along the whole curve.

Reusable code lives in [`pes_tools.py`](pes_tools.py). Put anything you'd call twice there, because it merges cleanly in git and notebooks don't. Results go to `results/` as CSV and plots to `figures/`, so they're shared even though notebook outputs are stripped on commit.

In [ ]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pes_tools import (
    CHEMICAL_ACCURACY, HARTREE_TO_KCAL_MOL,
    build_problem, hf_energy, exact_energy, vqe_energy,
)

warnings.filterwarnings("ignore", module="scipy")  # SparseEfficiencyWarning noise
RESULTS, FIGURES = Path("results"), Path("figures")

## 0. Sanity check: one geometry

HeH$^+$ has 2 electrons (He gives 2, H gives 1, and the +1 charge removes one), so in `sto3g` it is again 2 spatial orbitals → 4 qubits, just like H$_2$.

> ⚠️ **Gotcha found while setting up:** the tutorial's unfiltered `NumPyMinimumEigensolver` gives the wrong "exact" answer for HeH$^+$ (≈0.16 Ha too low). The qubit Hamiltonian covers every electron count, and the lowest state overall has *extra* electrons. `exact_energy` in `pes_tools.py` filters to the correct particle number. Worth a sentence in the write-up.

In [ ]:
problem = build_problem(0.774)
print("spatial orbitals:", problem.num_spatial_orbitals, "| particles (α, β):", problem.num_particles)
print("nuclear repulsion:", problem.nuclear_repulsion_energy)

point = vqe_energy(problem)
print(f"HF    {hf_energy(problem): .6f} Ha")
print(f"VQE   {point.energy: .6f} Ha   ({point.num_qubits} qubits, {point.num_parameters} params)")
print(f"exact {exact_energy(problem): .6f} Ha")

## 1. Sweep the bond distance

**TODO (decide together):**
- What range? Short distances are very repulsive, and long distances probe dissociation. How far out before the curve flattens?
- How many points, and should the grid be denser near the minimum?
- Start each point from zeros (= HF), or warm-start from the previous point's `optimal_point`? Compare iteration counts (`cost_function_evals`).

The grid below is a deliberately coarse placeholder.

In [ ]:
distances = np.linspace(0.5, 3.0, 6)  # TODO: choose the real grid

rows, prev_params = [], None
for d in distances:
    t0 = time.perf_counter()
    prob = build_problem(float(d))
    vqe = vqe_energy(prob, initial_point=None)  # TODO: try initial_point=prev_params
    rows.append({
        "distance_A": d,
        "hf": hf_energy(prob),
        "vqe": vqe.energy,
        "exact": exact_energy(prob),
        "vqe_evals": vqe.cost_function_evals,
        "seconds": time.perf_counter() - t0,
    })
    prev_params = vqe.optimal_point

pes = pd.DataFrame(rows)
pes["vqe_error"] = (pes.vqe - pes.exact).abs()
pes.to_csv(RESULTS / "heh_plus_pes_sto3g.csv", index=False)
pes

## 2. Plot the curve

In [ ]:
fig, (ax, ax_err) = plt.subplots(2, 1, figsize=(8, 7), sharex=True, height_ratios=[3, 1])
ax.plot(pes.distance_A, pes.hf, "o--", color="#9e9e9e", label="Hartree-Fock")
ax.plot(pes.distance_A, pes.exact, "-", color="#2ca02c", label="Exact (filtered)")
ax.plot(pes.distance_A, pes.vqe, "x", color="#1f77b4", ms=9, label="VQE (UCCSD)")
ax.set_ylabel("Total energy (Ha)")
ax.set_title("HeH$^+$ potential energy surface (sto3g)")
ax.legend()

ax_err.semilogy(pes.distance_A, pes.vqe_error, "o-", color="#1f77b4")
ax_err.axhline(CHEMICAL_ACCURACY, color="red", ls=":", label="chemical accuracy")
ax_err.set_xlabel("He–H distance (Å)")
ax_err.set_ylabel("|VQE − exact| (Ha)")
ax_err.legend()
fig.tight_layout()
fig.savefig(FIGURES / "heh_plus_pes_sto3g.png", dpi=150)

## 3. Equilibrium bond length

**TODO:** the grid minimum is only as good as the grid spacing. Refine it with a denser grid near the minimum, a polynomial/spline fit, or a 1-D optimiser (e.g. `scipy.optimize.minimize_scalar`) wrapped around `vqe_energy`. Compare with the experimental value (look it up and cite it).

In [ ]:
# TODO

## 4. Dissociation

**TODO:** what does HeH$^+$ fall apart into: He + H$^+$, or He$^+$ + H? Compute the energies of the separated fragments (each one with `PySCFDriver`, with the right charge and spin) and compare them with the long-distance end of the curve. Then estimate the dissociation energy $D_e = E(\infty) - E(r_{eq})$ in Ha and kcal/mol.

In [ ]:
# TODO

## 5. Does the ansatz stay accurate everywhere?

**TODO:** look at the error panel. Where is VQE furthest from exact, and why? (Hint: compare how far HF is from exact along the curve.) Things to try: other optimisers, warm starts, a hardware-efficient ansatz (`qiskit.circuit.library.efficient_su2`) instead of UCCSD, a bigger basis (`6-31g`), or a noisy `qiskit_aer` estimator.

In [ ]:
# TODO

## 6. Conclusions

**TODO:** equilibrium bond length, dissociation products and energy, where the ansatz succeeds or fails, and what we'd do next.